# Mock-mode tour

This notebook runs AEC Model Bridge in mock mode: no Revit, no Windows. It calls the same `call_tool` handler an MCP client reaches, in this order:

1. `revit_get_context`: what the server is connected to.
2. `search_tools`: find a tool by keyword.
3. `snapshot_take`: save a snapshot of the (mock) model.
4. `snapshot_query`: compute counts from the saved snapshot.
5. `snapshot_compute`: answer a grouped question in one call.
5. `plan_actions`: propose a change. The tour stops here, because only a person can approve a plan.

Run it from `packages/mcp-server-revit` after `uv sync --all-extras`, for example with `uv run --with jupyter jupyter lab ../../examples/mock-tour.ipynb`. Each run writes to a new temporary workspace folder.

In [ ]:
import json
import os
import tempfile

# Settings are read when the package is imported, so set them first.
os.environ["MCP_REVIT_MODE"] = "mock"
os.environ["MCP_REVIT_WORKSPACE_DIR"] = tempfile.mkdtemp(prefix="aec-mock-tour-")

from revit_mcp_server import mcp_server
from revit_mcp_server.registry_factory import build_registry
from revit_mcp_server.tool_profiles import load_tool_profile

# The same wiring the server does at start-up (mcp_server.main), without stdio.
registry, approval, jobs, _modules, workspace = build_registry()
registry.freeze_tool_profile(load_tool_profile())
mcp_server.registry = registry
mcp_server.approval_provider = approval
mcp_server.job_manager = jobs
mcp_server.workspace_monitor = workspace


async def call(name, arguments=None):
    """Call a tool the way an MCP client does and return its JSON result."""
    out = await mcp_server.call_tool(name, arguments or {})
    if getattr(out, "isError", False):
        return out.structuredContent  # {"ok": false, "error": {"kind": ..., "message": ...}}
    text = out[0].text
    return json.loads(text.split("Result:\n", 1)[1])


print("workspace:", os.environ["MCP_REVIT_WORKSPACE_DIR"])
print("tools listed:", len(await mcp_server.list_tools()))

In [ ]:
context = await call("revit_get_context")
assert context["mock"] is True
print(context["message"])
print("approval mode:", context["server"]["approval_mode"])

In [ ]:
found = await call("search_tools", {"query": "snapshot"})
for tool in found["tools"]:
    print(f"{tool['name']:32} read_only={tool['read_only']}")

In [ ]:
snapshot = await call("snapshot_take")
snapshot_id = snapshot["snapshot_id"]
print(snapshot_id, "elements:", snapshot["elements_count"])

In [ ]:
from collections import Counter

everything = await call("snapshot_query", {"snapshot_id": snapshot_id, "filter": {}})
by_category = Counter(element["category"] for element in everything["elements"])
for category, count in sorted(by_category.items()):
    print(f"{category:20} {count}")

walls = await call("snapshot_query", {"snapshot_id": snapshot_id, "filter": {"category": "OST_Walls"}})
wall = walls["elements"][0]
print("first wall:", wall["element_id"], wall["type_name"])

In [ ]:
# One call answers a grouped question: door count and marks per level.
summary = await call("snapshot_compute", {"snapshot_id": snapshot_id, "steps": [
    {"name": "doors", "where": {"field": "category", "value": "OST_Doors"}, "join": ["level"]},
    {"from": "$doors", "group_by": ["level.type_name"],
     "aggregate": {"doors": "count", "marks": {"op": "list", "field": "params.Mark"}}},
]})
for row in summary["rows"]:
    print(row)

## Propose a change

A model write never runs directly. `plan_actions` records the proposed actions as a plan in state `pending`, with a preview and a hash of exactly what will run.

In [ ]:
plan = await call("plan_actions", {"actions": [{
    "tool": "revit_set_parameter_value",
    "arguments": {"element_id": wall["element_id"], "parameter_name": "Comments", "value": "Checked in the mock tour"},
}]})
assert plan["state"] == "pending"
print(plan["plan_id"], plan["state"], plan["plan_hash"])
print(json.dumps(plan["preview"]["rows"], indent=2))

In [ ]:
# Calling the write tool directly is refused by the approval gate.
direct = await call("revit_set_parameter_value",
                    {"element_id": wall["element_id"], "parameter_name": "Comments", "value": "no plan"})
assert direct["error"]["kind"] == "approval_blocked"
print(direct["error"]["message"])

## Stop: a person approves

The plan stays `pending` until a person approves or rejects it in the Revit panel or with the `aec-model-bridge-approve` command. AI clients cannot call the approve tool. After approval, `execute_plan` runs exactly the approved actions. See `docs/security.md` for the approval gate.